In [ ]:
# E06 — pulls the code from GitHub so a run is reproducible from a commit SHA
import os, subprocess, sys, shutil, time
REPO = "https://github.com/Ahmadrezanourozii/Automatic-detection-of-diabetic-retinopathy-and-grading-of-diabetic-macular-edema-using-CNN.git"
COMMIT = "52d5ef48175a84275e22cf4910cf1db494e48662"
WORK = "/kaggle/working/repo"
if os.path.isdir(WORK):
    shutil.rmtree(WORK)
subprocess.run(["git", "clone", "--quiet", REPO, WORK], check=True)
if COMMIT and COMMIT != "HEAD":
    subprocess.run(["git", "-C", WORK, "checkout", "--quiet", COMMIT], check=True)
sha = subprocess.check_output(["git", "-C", WORK, "rev-parse", "HEAD"]).decode().strip()
print("CODE COMMIT", sha)
print(subprocess.check_output(["git", "-C", WORK, "log", "-1", "--pretty=%s"]).decode().strip())


In [ ]:
import os
for d in sorted(os.listdir("/kaggle/input")):
    n = sum(len(f) for _, _, f in os.walk(f"/kaggle/input/{d}"))
    print(f"{d:55s} {n:7d} files")
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")


In [ ]:
# The pool does not always honour the pinned accelerator. A P100 is sm_60 and the
# preinstalled torch cu128 build ships sm_70+ kernels only, so every CUDA call fails.
# Rather than lose the run, install a torch that supports this device (ISSUES.md §9).
import subprocess, sys, torch
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print(f"GPU {name}  sm_{cap[0]}{cap[1]}  torch {torch.__version__}")
ok = True
try:
    (torch.zeros(8, 8, device="cuda") @ torch.zeros(8, 8, device="cuda")).sum().item()
    print("kernels execute fine on this device")
except Exception as e:
    ok = False
    print("UNUSABLE:", e)
if not ok:
    print("installing a torch build that supports this GPU ...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch==2.5.1", "torchvision==0.20.1",
                    "--index-url", "https://download.pytorch.org/whl/cu121"], check=False)
    print("installed -- src/train.py runs in a subprocess so it picks up the new build")


In [ ]:
import subprocess, sys, os, time
RUN_ID = "E06"
OUT = f"/kaggle/working/{RUN_ID}"
os.makedirs(OUT, exist_ok=True)
LOG = f"{OUT}/train.log"

cmd = [sys.executable, "-u", "/kaggle/working/repo/src/train.py",
       "--datasets", "/kaggle/input",
       "--splits", "/kaggle/working/repo/data/splits/dev_v1.json",
       "--run-id", RUN_ID, "--out", OUT,
       "--cache", "/kaggle/temp/cache560",
       "--channels-last", "--resume",
       "--folds", "0,1,2,3,4", "--epochs", "25", "--size", "448", "--batch", "16", "--backbone", "densenet121", "--head", "ordinal", "--workers", "2", "--tta", "--pretrain-corpora", "EyePACS", "--pretrain-epochs", "4", "--hypothesis", "eyepacs-pretraining-then-finetune-on-dev-pool"]
print(" ".join(cmd), flush=True)

# tee to the log file AND to the notebook output, so a killed session still leaves a log
t0 = time.time()
with open(LOG, "a") as f:
    f.write(f"\n===== launched {time.strftime('%Y-%m-%d %H:%M:%S')} =====\n")
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         bufsize=1, cwd="/kaggle/working/repo")
    for line in p.stdout:
        print(line, end="", flush=True)
        f.write(line); f.flush()
    p.wait()
print(f"\nexit={p.returncode}  {time.time()-t0:.0f}s")


In [ ]:
# keep results.json, the log and the out-of-fold predictions; drop the heavy checkpoints
import glob, os, shutil, json
RUN_ID = "E06"
OUT = f"/kaggle/working/{RUN_ID}"
for p in glob.glob(f"{OUT}/ckpt_*.pt"):
    print("dropping", os.path.basename(p), f"{os.path.getsize(p)/1e6:.0f} MB")
    os.remove(p)
for p in sorted(glob.glob(f"{OUT}/*")):
    print(f"{os.path.getsize(p)/1e6:8.2f} MB  {os.path.basename(p)}")
rj = f"{OUT}/results.json"
if os.path.exists(rj):
    r = json.load(open(rj))
    print("\nrun", r["run_id"], "commit", r["commit"][:10],
          "| folds", [f["fold"] for f in r["folds"]])
    if "pooled_oof" in r:
        for k, v in r["pooled_oof"]["metrics"].items():
            print(f"  {k:22s} n={v['n']:5d} acc {v['accuracy']*100:5.1f}% "
                  f"floor {v['majority_floor']*100:5.1f}% QWK {v['qwk']:6.3f}")
